In [5]:
from datasets import load_dataset

ds_2024 = load_dataset("Reubencf/2024_events")
ds_2025 = load_dataset("Reubencf/2025_events")

from datasets import concatenate_datasets

combined_events = concatenate_datasets([
    ds_2024["train"],
    ds_2025["train"]
])

events = combined_events.map(
    lambda x: {"section": x["section"].lower()}
)

In [14]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from pprint import pprint
import re

# https://reference.langchain.com/python/langchain-chroma/vectorstores/Chroma

event_documents = []
event_ids = []

events_vector_store = Chroma(
    persist_directory="./chroma_db",
    collection_name="events",
    embedding_function=OllamaEmbeddings(model="nomic-embed-text")
)

ei = 0;

for event in events:
    date_prefix = f"{event['month']} {event['day']}, {event['year']}: "
    text = f"{date_prefix}\n\nInstruction: {event['instruction']}\n\nContent: {event['content']}"

    parts = re.split(r'(### .+)', event["response"])

    sections = []
    current_title = "Introduction"
    current_content = ""

    for part in parts:
        if part.startswith("###"):
            if current_content:
                sections.append((current_title, current_content.strip()))
            current_title = part.replace("###", "").strip()
            current_title = current_title.replace("**", "").strip()
            
            current_content = ""
        else:
            current_content += part

        if current_content:
            current_content = current_content.replace("\n\n#", "").strip()
            current_content = current_content.replace("\n*", "").strip()
            current_content = current_content.replace("**", "").strip()
            current_content = current_content.replace("\n\n---", "").strip()

            sections.append((current_title, current_content.strip()))

            if ei < 10:
                print("\n\nSection:")
                print(sections[-1]) # print the last element

    event_documents.append(Document(
        page_content = text, # this will be converted to document
        metadata={
            "month": event["month"],
            "day": event["day"],
            "year": event["year"],
            "section": event["section"]
        })
    )
    
    event_ids.append(f"e{ei}")

    ei=ei+1

    # if ei > 3:
    #     break

print("Generating embeddings and storing...")
events_vector_store.reset_collection()
batch_size = 100

for i in range(0, len(event_documents), batch_size):
    doc_batch = event_documents[i:i+batch_size]
    id_batch = event_ids[i:i+batch_size]
    
    events_vector_store.add_documents(ids=id_batch, documents=doc_batch)

    print(f"\r{i + len(doc_batch)} / {len(event_documents)} [ {round((i + len(doc_batch)) / len(event_documents) * 100)}% ]", end="")

print("\nStoring...")
results = events_vector_store.get(limit=3)
print("Done!")




Section:
('Introduction', 'The announcement by the Israel Defense Forces (IDF) on January 1, 2024, that it was withdrawing five brigades from the Gaza Strip and entering a "different mode of operations" marked a pivotal strategic transition in the Israel-Hamas war. This shift, widely characterized by military analysts and Israeli officials as the beginning of "Phase Three" of the conflict, signified an end to the large-scale, high-intensity ground maneuver that had defined the initial weeks of the invasion, moving toward a protracted, low-intensity, intelligence-driven campaign of targeted raids and limited incursions.\n\nThis transition was a complex response to a combination of internal economic strain, military objectives being met in parts of the territory, and sustained international diplomatic pressure, particularly from the United States, to reduce the immense toll on the civilian population in the Gaza Strip. The "different mode of operations" was not an end to the war, but r